<a href="https://colab.research.google.com/github/swapnilsurdi/practice_agentic/blob/main/autonomous_it_support_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Autonomous IT Support Agent**

This is a **Level 1 IT Incident Responder**.

requires: `OPENAI_API_KEY`

---

**Objective:** You are building an AI agent that acts as the "first responder" for server incidents. It must:

1. **Investigate:** Check server health and logs when a user reports an issue.
2. **Act:** If CPU is critical (>90%), it should **Restart** the service.
3. **Escalate:** If the issue is complex or logs show a major problem it should **Escalate** to a human.


In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [2]:
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# Initialize Client

==========================================
## PART 1: DEFINE THE TOOLS (BUSINESS LOGIC)
==========================================

In [3]:
# --- Already implement tool 1: Check Health ---
def get_server_health(server_id: str) -> str:
    """Returns CPU and Memory usage for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)


In [4]:
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the last N lines of logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Different logs for different servers to trigger different agent behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

---
### ----- Implement code below -----
---


In [43]:
# --- TASK 1: Implement the Restart Tool ---
def restart_service(server_id: str) -> str:
    """
    1. Print a message saying "-> TOOL: Restarting service..."
    2. Return a JSON string confirming the restart was successful.
       Example return: '{"status": "success", "message": "Server restarted successfully"}'
    """

    print("TOOL: Restarting service: ", server_id)
    return '{"status": "success", "message": "Server restarted successfully"}'


In [44]:
# --- TASK 2: Implement the Escalation Tool ---
def escalate_to_engineer(summary: str) -> str:
    """
    1. Print a message saying "-> TOOL: Escalating to human..."
    2. Return a JSON string confirming the ticket was created.
    """
    print("TOOL: Escalating to human: ", summary)
    return '{"status": "success", "message": "Ticket created successfully"}'


In [11]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

==========================================
## PART 2: DEFINE THE AGENT SCHEMA
==========================================


In [42]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Restarts the service",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail or the error is unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary": {"type": "string", "description": "The summary of the issue for human to understand."},
                },
                "required": ["summary"]
            }
        }
    }
]


 ==========================================
## PART 3: THE AGENT EXECUTION LOOP
 ==========================================


In [39]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate server issues. "
                                      "If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]
    turns = 0 # added safety for not consuming too many tokens.
    while turns < 5:
        turns+=1
        print("\n[AI Thinking...]", turns)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            print("tool_calls", response_msg.tool_calls)
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)
                message = {
                    "role": "tool",
                    "tool_call_id": tool_call.id, # What goes here?
                    "name": tool_call.function.name,         # What goes here?
                    "content": ""       # What goes here?
                }
                tool_output = ""
                if function_to_call:
                    # Execute the function
                    try:
                        tool_output = function_to_call(**func_args)
                    except Exception as e:
                        print(f"Error calling {func_name}: {e}")
                        tool_output += f"Failed: {func_name}: {e}"

                else:
                    tool_output = f"Unknown Function: {func_name}"
                message["content"] = tool_output
                messages.append(message)
        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            break

 ==========================================
## PART 4: TEST SCENARIOS
 ==========================================


In [40]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")


--- New Incident: The payment-server-01 is extremely slow and timing out. ---

[AI Thinking...] 1
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_05h5tIfNiXDM8Mj5yQviFyXJ', function=Function(arguments='{"server_id": "payment-server-01"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_vNVEC8neYwDXd11JsHf34UhM', function=Function(arguments='{"server_id": "payment-server-01", "lines": 20}', name='fetch_recent_logs'), type='function')]
-> TOOL: Checking health for payment-server-01...
-> TOOL: Fetching last 20 log lines for payment-server-01...

[AI Thinking...] 2
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_yPLVbLiA4lYdShBtdIFo5cU5', function=Function(arguments='{"server_id":"payment-server-01"}', name='restart_service'), type='function')]
TOOL: Restarting service:  payment-server-01

[AI Thinking...] 3

[FINAL RESPONSE]: I have restarted the payment-server-01 as it was experiencing high CPU usage (98%) and thread poo

In [33]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird) | nothing weird in logs
run_it_agent("Something is wrong with db-node-02")


--- New Incident: Something is wrong with db-node-02 ---

[AI Thinking...] 1
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_gzoCl5og9yEA3c8Y3eFZ97r0', function=Function(arguments='{"server_id": "db-node-02"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_vVge7szF2Jzq7cKioXMJKaIk', function=Function(arguments='{"server_id": "db-node-02", "lines": 10}', name='fetch_recent_logs'), type='function')]
-> TOOL: Checking health for db-node-02...
-> TOOL: Fetching last 10 log lines for db-node-02...

[AI Thinking...] 2

[FINAL RESPONSE]: The server `db-node-02` appears to be healthy with a CPU usage of 12% and memory usage of 60%. The recent logs do not show any critical dependency errors or issues; they indicate normal operations such as backups and user queries being executed successfully. If users continue to experience issues, it might be due to an external factor not currently logged or affecting performance metrics.


In [34]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")


--- New Incident: Users are reporting login failures on auth-service-03. ---

[AI Thinking...] 1
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_OQkFSw6GQrtPpgQHPslaa4MR', function=Function(arguments='{"server_id": "auth-service-03"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_Csnl1Eplocv1b39d01izSW05', function=Function(arguments='{"server_id": "auth-service-03", "lines": 10}', name='fetch_recent_logs'), type='function')]
-> TOOL: Checking health for auth-service-03...
-> TOOL: Fetching last 10 log lines for auth-service-03...

[AI Thinking...] 2
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_PMAnKUpT0B3ediEXXuDtnumi', function=Function(arguments='{"server_id":"auth-service-03"}', name='restart_service'), type='function')]
TOOL: Restarting service:  auth-service-03

[AI Thinking...] 3

[FINAL RESPONSE]: The `auth-service-03` server was experiencing a memory issue caused by a memory leak which led to the applicat

In [35]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Search isn't working. Can you check search-index-09? ---

[AI Thinking...] 1
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_HuZjh0b0KHo3gCzzVcpR2El8', function=Function(arguments='{"server_id": "search-index-09"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_FZThrqXNh3rrI6MAm4JTwQcv', function=Function(arguments='{"server_id": "search-index-09", "lines": 20}', name='fetch_recent_logs'), type='function')]
-> TOOL: Checking health for search-index-09...
-> TOOL: Fetching last 20 log lines for search-index-09...

[AI Thinking...] 2
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_5mtErgOCoA9dbYG3YHnZhvi8', function=Function(arguments='{"summary":"The server \'search-index-09\' is experiencing search failures due to critical dependency issues. Logs indicate a \'Connection refused\' error to the Elastic cluster at \'elastic-cluster-main:9200\', showing that the dependency is unreachable. A restart will 

In [36]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")


--- New Incident: Check frontend-node-04 just to be safe. ---

[AI Thinking...] 1
tool_calls [ChatCompletionMessageFunctionToolCall(id='call_gtrLmTh0ld9GkDD7F4zLTJYi', function=Function(arguments='{"server_id": "frontend-node-04"}', name='get_server_health'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_sY6Rxf5S074bOgPTj5J4V3ug', function=Function(arguments='{"server_id": "frontend-node-04", "lines": 50}', name='fetch_recent_logs'), type='function')]
-> TOOL: Checking health for frontend-node-04...
-> TOOL: Fetching last 50 log lines for frontend-node-04...

[AI Thinking...] 2

[FINAL RESPONSE]: The server "frontend-node-04" is functioning normally with a CPU usage of 25% and memory usage of 30%, both well within safe limits. Recent logs show no errors, only successful requests and health checks. No further action is required at this time.
